# Análise do modelo de classificação de coral-sol, segunda e terceira versão, com o dataset PUC-Legado

Autor:  Viviane da Rosa Sommer

Atualizado: 21/02/2024

## Objetivo:
Notebook para fazer a predição das imagens do dataset PUC-Legado pelo modelo V2 e V3 de classificação de coral-sol, para analisar o desempenho do mesmo.

## Importações Necessárias

In [ ]:
'''!pip install opencv-python-headless'''
import numpy as np
import cv2
import glob
import os
import keras
import matplotlib.pyplot as plt
from ultralytics import YOLO
from matplotlib.patches import Rectangle
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
from tensorflow.keras.models import Model
from tensorflow.keras.utils import img_to_array, load_img

## Declaração de Constantes e Modelos

In [ ]:
INPUT_DIRECTORY_POSITIVE_IMAGES = "../Coral_Note/puc_dataset/PUC - Coral - Positivo" 
INPUT_DIRECTORY_NEGATIVE_IMAGES = "../Coral_Note/puc_dataset/PUC - Coral - Negativo" 
V1_SUBIMAGE_SIZE = 128
BATCH_SIZE = 16 

model_v1 = keras.models.load_model('Pesos/best_weights_v1.keras') 
model_v2 = YOLO("runs/classify/train-patience100-500epochs/weights/best.pt")
true_labels = [] 
predicted_labels_v1 = [] 
predicted_labels_v2 = [] 

## Funções Necessárias

In [ ]:
def read_image(filename: str) -> np.ndarray:
    """
    Reads an image from a file and converts it into a NumPy array.

    Args:
        filename (str): Path to the image file.

    Returns:
        np.ndarray: The cropped image as a NumPy array.
    """
    try:
        image = load_img(filename)
        image = img_to_array(image)
        height, width, _ = image.shape
        mid = height // 2
        top = max(0, mid - int(0.34 * height))
        bottom = min(height, mid + int(0.34 * height))
        cropped_image = image[top:bottom, :]
        return cropped_image
    except Exception as e:
        print(f"Error reading image {filename}: {e}")
        return None
 
def pad_sub_image(sub_image: np.ndarray, size: int) -> np.ndarray:
    """
    Pads a sub-image to the specified size.

    Args:
        sub_image (np.ndarray): The sub-image to be padded.
        size (int): The desired size of the padded sub-image.

    Returns:
        np.ndarray: The padded sub-image filled with black (zeros).
    """
    padded_image = np.zeros((size, size, 3), dtype=np.uint8)
    height, width, _ = sub_image.shape
    padded_image[:height, :width, :] = sub_image.astype(np.uint8)
    return padded_image
 
def divide_image(image: np.ndarray, size: int) -> tuple[list[np.ndarray], list[tuple[int, int]]]:
    """
    Divides an image into sub-images of a specified size.

    Args:
        image (np.ndarray): The input image.
        size (int): The size of each sub-image.

    Returns:
        tuple[list[np.ndarray], list[tuple[int, int]]]: A tuple containing the list of sub-images and their coordinates.
    """
    sub_images = []
    coordinates = []
    height, width, _ = image.shape

    for y in range(0, height, size):
        for x in range(0, width, size):
            sub_image = image[y:min(y + size, height), x:min(x + size, width)]
            sub_image = pad_sub_image(sub_image, size)
            sub_images.append(sub_image)
            coordinates.append((x, y))

    return sub_images, coordinates


def preprocess_image(image: np.ndarray, size: int = None) -> np.ndarray:
    """
    Preprocesses an image for use by the model.

    Args:
        image (np.ndarray): The input image.
        size (int, optional): The desired size of the preprocessed image. Defaults to None.

    Returns:
        np.ndarray: The preprocessed image.
    """
    if size:
        image = cv2.resize(image, (size, size))
    image = image.astype(np.float32) / 255.0
    return image
    
 
def batch_predict(model: Model, data: list[np.ndarray], batch_size: int) -> np.ndarray:
    """
    Performs batch predictions to avoid memory issues.

    Args:
        model (Model): The Keras model for prediction.
        data (list[np.ndarray]): The input data for prediction.
        batch_size (int): The size of each batch.

    Returns:
        np.ndarray: Predictions for all input data.
    """
    predictions = []
    for i in range(0, len(data), batch_size):
        batch = np.array(data[i:i + batch_size])
        try:
            batch_predictions = model.predict(batch, verbose=0)
            predictions.extend(batch_predictions)
        except Exception as e:
            print(f"Error in batch prediction: {e}")
            continue
    return np.array(predictions)
    
 
def plot_results(image: np.ndarray, results_v1: np.ndarray, results_v2: np.ndarray,
                 coordinates_v1: list[tuple[int, int]], true_label: str, predicted_label_v2: str, v2_score: float) -> None:
    """
    Plots the original image, Model V1 results, and Model V2 results with prediction scores.

    Args:
        image (np.ndarray): The input image.
        results_v1 (np.ndarray): Predictions from Model V1.
        results_v2 (np.ndarray): Predictions from Model V2.
        coordinates_v1 (list[tuple[int, int]]): Coordinates for Model V1 sub-images.
        true_label (str): The true label of the image.
        predicted_label_v2 (str): The predicted label from Model V2.
        v2_score (float): The confidence score of the predicted label from Model V2.
    """
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(24, 8))

    ax1.imshow(image.astype(np.uint8))
    ax1.set_title(f"True label: {true_label}", fontsize=14)
    ax1.axis('off')

    ax2.imshow(image.astype(np.uint8))
    for x, y in coordinates_v1:
        rect = Rectangle((x, y), 128, 128, linewidth=1, edgecolor='blue', facecolor='none')
        ax2.add_patch(rect)
    for idx, (x, y) in enumerate(coordinates_v1):
        if results_v1[idx]:
            rect = Rectangle((x, y), 128, 128, linewidth=2, edgecolor='red', facecolor='none')
            ax2.add_patch(rect)
    ax2.set_title("Model V2 Predictions", fontsize=14)
    ax2.axis('off')

    ax3.imshow(image.astype(np.uint8))
    ax3.set_title(f"Model V3 Prediction: {predicted_label_v2} ({v2_score:.2f})", fontsize=14)
    ax3.axis('off')

    plt.tight_layout()
    plt.show()
 
def evaluate_classification(true_labels: list[str], predicted_labels_v1: list[str], predicted_labels_v2: list[str]) -> None:
    """
    Evaluates classification performance by generating confusion matrices and classification reports.

    Args:
        true_labels (list[str]): The true labels.
        predicted_labels_v1 (list[str]): Predictions from Model V1.
        predicted_labels_v2 (list[str]): Predictions from Model V2.
    """
    true_labels_binary = [1 if label == 'Positive' else 0 for label in true_labels]
    predicted_labels_binary_v1 = [1 if label == 'Positive' else 0 for label in predicted_labels_v1]
    predicted_labels_binary_v2 = [1 if label == 'Positive' else 0 for label in predicted_labels_v2]

    conf_matrix_v1 = confusion_matrix(true_labels_binary, predicted_labels_binary_v1)
    disp_v1 = ConfusionMatrixDisplay(confusion_matrix=conf_matrix_v1, display_labels=['Negative', 'Positive'])
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    disp_v1.plot(cmap=plt.cm.Blues, ax=ax1)
    ax1.set_title("Confusion Matrix - V2 Model")

    conf_matrix_v2 = confusion_matrix(true_labels_binary, predicted_labels_binary_v2)
    disp_v2 = ConfusionMatrixDisplay(confusion_matrix=conf_matrix_v2, display_labels=['Negative', 'Positive'])
    disp_v2.plot(cmap=plt.cm.Blues, ax=ax2)
    ax2.set_title("Confusion Matrix - V3 Model")

    plt.tight_layout()
    plt.show()

    print("Classification Report - V2 Model:")
    print(classification_report(true_labels_binary, predicted_labels_binary_v1, target_names=['Negative', 'Positive']))
    print("\nClassification Report - V3 Model:")
    print(classification_report(true_labels_binary, predicted_labels_binary_v2, target_names=['Negative', 'Positive']))

## Processamento das imagens - Casos Positivos

In [ ]:
for filename in glob.glob(f"{INPUT_DIRECTORY_POSITIVE_IMAGES}/**", recursive=True):
    try:
        if not os.path.isfile(filename):
            continue

        if filename.split(".")[-1] == "json":
            continue

        image = read_image(filename)
        if image is None:
            continue

        # Model V1: Process sub-images and predict
        sub_images_v1, coordinates_v1 = divide_image(image, V1_SUBIMAGE_SIZE)
        sub_images_v1 = [preprocess_image(img, V1_SUBIMAGE_SIZE) for img in sub_images_v1]
        sub_images_v1 = np.array(sub_images_v1)
        results_v1 = batch_predict(model_v1, sub_images_v1, BATCH_SIZE)
        binary_results_v1 = results_v1 >= 0.5  # Boolean array for sub-image predictions

        # Determine Model V1 predicted label
        predicted_label_v1 = 'Positive' if binary_results_v1.any() else 'Negative'

        # Model V2: Full image predictions
        results_v2 = model_v2.predict(source=filename, save=False)
        print(results_v2)
        class_probabilities = results_v2[0].probs.data.cpu().numpy()
        predicted_label_index = np.argmax(class_probabilities)  # Index of the class with the highest probability
        v2_score = class_probabilities[predicted_label_index]  # Confidence score for the predicted label
        predicted_label_v2 = 'Positive' if predicted_label_index == 1 else 'Negative'

        # Determine true label
        true_label = 'Positive'

        # Store results
        true_labels.append(true_label)
        predicted_labels_v1.append(predicted_label_v1)
        predicted_labels_v2.append(predicted_label_v2)

        # Visualize results
        plot_results(
            image=image,
            results_v1=binary_results_v1,
            results_v2=class_probabilities,
            coordinates_v1=coordinates_v1,
            true_label=true_label,
            predicted_label_v2=predicted_label_v2,
            v2_score=v2_score
        )

    except Exception as e:
        print(f"Error processing file {filename}: {e}")
        continue

## Processamento das imagens - Casos Negativos

In [ ]:
for filename in glob.glob(f"{INPUT_DIRECTORY_NEGATIVE_IMAGES}/**", recursive=True):
    try:
        if not os.path.isfile(filename):
            continue

        image = read_image(filename)
        if image is None:
            continue

        # Model V1: Process sub-images and predict
        sub_images_v1, coordinates_v1 = divide_image(image, V1_SUBIMAGE_SIZE)
        sub_images_v1 = [preprocess_image(img, V1_SUBIMAGE_SIZE) for img in sub_images_v1]
        sub_images_v1 = np.array(sub_images_v1)
        results_v1 = batch_predict(model_v1, sub_images_v1, BATCH_SIZE)
        binary_results_v1 = results_v1 >= 0.5  # Boolean array for sub-image predictions

        # Determine Model V1 predicted label
        predicted_label_v1 = 'Positive' if binary_results_v1.any() else 'Negative'

        # Model V2: Full image predictions
        results_v2 = model_v2.predict(source=filename, save=False)
        class_probabilities = results_v2[0].probs.data.cpu().numpy()
        predicted_label_index = np.argmax(class_probabilities)  # Index of the class with the highest probability
        v2_score = class_probabilities[predicted_label_index]  # Confidence score for the predicted label
        predicted_label_v2 = 'Positive' if predicted_label_index == 1 else 'Negative'

        # Determine true label
        true_label = 'Negative'

        # Store results
        true_labels.append(true_label)
        predicted_labels_v1.append(predicted_label_v1)
        predicted_labels_v2.append(predicted_label_v2)

        # Visualize results
        plot_results(
            image=image,
            results_v1=binary_results_v1,
            results_v2=class_probabilities,
            coordinates_v1=coordinates_v1,
            true_label=true_label,
            predicted_label_v2=predicted_label_v2,
            v2_score=v2_score
        )

    except Exception as e:
        print(f"Error processing file {filename}: {e}")
        continue

## Métricas dos resultados

In [ ]:
evaluate_classification(true_labels, predicted_labels_v1, predicted_labels_v2) 

In [ ]:
!jupyter nbconvert --to html analize_model_comparacao.ipynb